# Cookie Cats A/B Test: Gate Placement and Player Retention

**Business question:** Cookie Cats is a mobile puzzle game where players hit a "gate" that forces them to wait or pay to continue. This experiment tests whether moving the first gate from **level 30** to **level 40** changes player retention.

This notebook follows a structured A/B testing workflow: design the test, check data quality, analyze the primary metric, validate the result with a second method, check a guardrail metric, and quantify what the test could detect before making a recommendation.

## 1. Experiment Design

- **Hypothesis:** Moving the gate from level 30 to level 40 changes 7-day retention.
- **Primary metric:** `retention_7` — the most business-relevant metric, since it measures whether players stay engaged over a full week.
- **Secondary metric:** `retention_1`.
- **Guardrail / engagement metric:** `sum_gamerounds` — more rounds generally means more ad and purchase exposure, so this checks for unintended side effects on engagement.
- **Significance level:** α = 0.05
- **Power:** 0.8
- **Test type:** two-sided

**Note:** This is a retrospective analysis of an already-collected dataset, so the design above is stated up front as if pre-registered, but it was not literally locked in before the experiment ran. The dataset also has no timestamps, so novelty effects and peeking cannot be checked here — see the Limitations section at the end.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.proportion import (
    proportions_ztest, confint_proportions_2indep, proportion_effectsize)
from statsmodels.stats.power import NormalIndPower

## 2. Load Data

In [ ]:
df = pd.read_csv("cookie_cats.csv")
df.head()

## 3. Data Quality Checks

Before trusting any result, confirm the dataset is clean and the random split behaved as expected.

In [ ]:
print(df.describe())
print()
df.info()
print()
print("Missing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

No null values and no duplicate rows, so we can proceed with the analysis.

### Sample Ratio Mismatch (SRM) Check

Before analyzing results, confirm the randomization produced the expected split. A skewed split (SRM) can indicate a bug in assignment or logging, which would invalidate any downstream comparison.

In [ ]:
counts = df["version"].value_counts().sort_index()
chi2, p_srm = stats.chisquare(counts, f_exp=[len(df) / 2] * 2)
print(counts.to_dict(), f"SRM p-value = {p_srm:.4f}")

To check whether our randomization logic failed, we ran a Chi-Square Goodness-of-Fit test as an SRM guardrail check. Using a strict industry-standard significance threshold of α = 0.001, the test yielded p = 0.0086. Because p > α (0.0086 > 0.001), we fail to reject the null hypothesis of a fair split and conclude that no severe sample ratio mismatch occurred. It is safe to proceed with the primary analysis.

## 4. Exploratory Data Analysis

In [ ]:
print(df["sum_gamerounds"].describe())
print("\nZero-round users:", (df["sum_gamerounds"] == 0).sum())
print("99th percentile of sum_gamerounds:", df["sum_gamerounds"].quantile(0.99))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 10))
sns.boxplot(data=df, x="version", y="sum_gamerounds", ax=ax[0])
sns.histplot(df[df.sum_gamerounds < 200], x="sum_gamerounds", hue="version",
             bins=50, ax=ax[1])
plt.show()

### Outlier Handling

The maximum value of `sum_gamerounds` (49,854) is far beyond the 99th percentile (~493) and is implausible for 14 days of play — most likely a bot or a logging error. We remove this single extreme outlier (values ≥ 40,000) before analyzing engagement. Retention metrics are unaffected by this decision since they are binary flags, not counts.

In [ ]:
df = df[df["sum_gamerounds"] < 40000]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 10))
sns.boxplot(data=df, x="version", y="sum_gamerounds", ax=ax[0])
sns.histplot(df[df.sum_gamerounds < 200], x="sum_gamerounds", hue="version",
             bins=50, ax=ax[1])
plt.show()

**Zero-round players:** 3,994 users played zero rounds (installed but never played). If we deleted these players, we would be ignoring people who installed the game and immediately uninstalled or gave up. We keep them in the dataset so the sample reflects all real-world installs — they are legitimate data points for retention, since they simply count as "not retained".

## 5. Retention Analysis (Primary and Secondary Metrics)

We compare retention rates between `gate_30` and `gate_40` using a two-proportion z-test for the point estimate and significance, and an unpooled confidence interval for the effect size.

In [ ]:
def compare_retention(metric):
    g = df.groupby("version")[metric].agg(x="sum", n="count")
    x, n = g["x"].astype(int).values, g["n"].values   # [gate_30, gate_40]
    rates = x / n

    z, p = proportions_ztest(x, n)                    # pooled SE (tests H0)
    low, high = confint_proportions_2indep(           # unpooled SE (CI)
        x[1], n[1], x[0], n[0], compare="diff", method="wald")

    return {"metric": metric,
            "gate_30": rates[0], "gate_40": rates[1],
            "diff (40-30)": rates[1] - rates[0],
            "CI_low": low, "CI_high": high,
            "z": z, "p_value": p}

results = pd.DataFrame([compare_retention(m) for m in ["retention_1", "retention_7"]])
results.round(4)

**Interpretation:**
- **Day 1 retention:** no significant change (p = 0.074 > 0.05). Moving the gate does not meaningfully affect whether players return the next day.
- **Day 7 retention (primary metric):** a clear negative effect (p = 0.0016 < 0.05), with the confidence interval for `gate_40 − gate_30` entirely below zero. Keeping the gate at level 30 yields higher 7-day retention.

Because `retention_7` is the pre-specified primary metric, this is the result that decides the test: **moving the gate to level 40 hurts long-term retention.**

### Visualization

In [ ]:
retention_summary = df.groupby("version")[["retention_1", "retention_7"]].mean().reset_index()
df_plot = retention_summary.melt(id_vars="version", var_name="Retention_Metric", value_name="Retention_Rate")
df_plot["Retention_Percentage"] = df_plot["Retention_Rate"] * 100

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=df_plot, x="Retention_Metric", y="Retention_Percentage", hue="version", palette="Set2")
plt.title("Retention Rates by Gate Placement", fontsize=14)
plt.ylabel("Retention Rate (%)")
plt.xlabel("Metric")
plt.ylim(0, 50)

for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{p.get_height():.2f}%",
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha="center", va="bottom", fontsize=10, xytext=(0, 3),
                    textcoords="offset points")

plt.show()

## 6. Robustness Check — Bootstrap

The z-test assumes a normal approximation. As a cross-check that does not rely on that assumption, we bootstrap the difference in `retention_7` between groups.

In [ ]:
rng = np.random.default_rng(42)

a = df.loc[df.version == "gate_30", "retention_7"].astype(int).values
b = df.loc[df.version == "gate_40", "retention_7"].astype(int).values

iterations = 5000
diffs = [
    rng.choice(b, len(b)).mean() - rng.choice(a, len(a)).mean()
    for _ in range(iterations)
]

ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
prob_gate40_better = np.mean(np.array(diffs) > 0)

print(f"Bootstrap 95% CI for difference (gate_40 - gate_30): [{ci_low:.4f}, {ci_high:.4f}]")
print(f"Probability that gate_40 is better than gate_30: {prob_gate40_better:.4f}")

The bootstrap confidence interval matches the z-test CI closely, and only 0.1% of bootstrap samples show `gate_40` performing better. Two independent methods — a parametric z-test and a non-parametric bootstrap — agree, which strengthens confidence in the conclusion.

## 7. Guardrail Metric — Engagement (`sum_gamerounds`)

`sum_gamerounds` is heavily right-skewed (see the EDA plots above), so we use the Mann-Whitney U test rather than a t-test, which assumes approximately normal data.

In [ ]:
rounds_30 = df.loc[df["version"] == "gate_30", "sum_gamerounds"]
rounds_40 = df.loc[df["version"] == "gate_40", "sum_gamerounds"]

stat, p_val = stats.mannwhitneyu(rounds_30, rounds_40, alternative="two-sided")

print(f"Mann-Whitney U Statistic: {stat:.2f}")
print(f"P-value: {p_val:.4f}")
print(f"Gate 30 Mean: {rounds_30.mean():.2f} | Median: {rounds_30.median():.1f}")
print(f"Gate 40 Mean: {rounds_40.mean():.2f} | Median: {rounds_40.median():.1f}")

With p ≈ 0.05 (just above α = 0.05), we fail to reject the null hypothesis: there is no statistically significant difference in player engagement (total game rounds) between `gate_30` and `gate_40`. The guardrail metric shows no clear harm to short-term engagement — the negative effect is specific to 7-day retention.

## 8. Power Analysis — Minimum Detectable Effect (MDE)

To understand what this sample size could reliably detect, we compute the minimum detectable effect (MDE) for `retention_7` at 80% power and α = 0.05.

In [ ]:
base = df.loc[df.version == "gate_30", "retention_7"].mean()
n_per_group = len(df) // 2

eff = NormalIndPower().solve_power(effect_size=None, nobs1=n_per_group,
                                   alpha=0.05, power=0.8, ratio=1)
# convert Cohen's h back to a proportion difference
p2 = np.sin(np.arcsin(np.sqrt(base)) + eff / 2) ** 2
print(f"Baseline retention_7: {base:.3f}")
print(f"MDE ≈ {p2 - base:.4f} ({(p2 - base) * 100:.2f} percentage points)")

With this sample size, the test could reliably detect a difference of roughly 0.7 percentage points in 7-day retention at 80% power. The observed effect (~0.8 points) is above this threshold, so the sample size was adequate to detect the effect found.

## 9. Conclusion & Recommendation

| Metric | gate_30 | gate_40 | Result |
|---|---|---|---|
| retention_1 | 44.82% | 44.23% | No significant difference (p = 0.074) |
| retention_7 (primary) | 19.02% | 18.20% | Significant decrease with gate_40 (p = 0.0016) |
| sum_gamerounds (guardrail) | mean 51.34 | mean 51.30 | No significant difference (p ≈ 0.05) |

**Recommendation: Keep the gate at level 30.** Moving it to level 40 produces a statistically significant drop in 7-day retention (the primary, pre-specified metric), confirmed independently by a bootstrap analysis, while producing no measurable benefit to day-1 retention or engagement.

**Business framing:** A 0.8 percentage point drop in 7-day retention, applied to a large install base, translates into meaningfully fewer weekly active players — for example, roughly 8,000 fewer retained users per 1,000,000 installs. This is a real cost with no observed offsetting benefit.

### Limitations
- The dataset has no timestamps, so novelty/primacy effects and the "peeking" problem could not be checked.
- This is a retrospective analysis; the experiment design above was not literally pre-registered before data collection.
- Only two retention snapshots (day 1, day 7) are available — intermediate dynamics (e.g., day 3) cannot be examined.
- The SRM check is borderline (p = 0.0086 against a strict α = 0.001 threshold); while not flagged as a failure, it is worth monitoring if this test were repeated.